<a href="https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09: Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Lane 1 (Ranking Signal Analysis) on the starter slice (30,000 rows, one row per content item). This notebook audits my own Week-5 model the way the live session audited the FlyRank research paper: the model is re-run under an honest split with a before/after comparison, every feature is checked against the label window, failures are looked at, and claims are rewritten in safe language. No warehouse needed.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the `hunting-leakage-and-validating` skill + `flyrank/flyrank-data`.

## 1. Two paper findings + my methodology questions

Read as: `docs/flyrank-seo-research-march-2026.pdf`. Two findings that carry broad recommendations, with a respectful, concrete methodology question for each. The paper itself discloses its standards (minimum 50 rows per bucket, negative and mixed results kept visible), so the questions below hold it to its own bar.

**Finding #1 (CONFIRMED): The Anatomy of Growing Content.** Growing pages are materially longer (3.2K vs 2.3K words, +37.6%) and younger (184 vs 230 days) than declining pages, over large cohorts (74,187 rising vs 45,272 falling). The recommendation is to expand thin pages that still earn impressions and to review aging pages before they drift.

My methodology question: **where does the label come from?** The direction label ("rising/falling impressions") and the reported health score are both products of the same trailing measurement window as the descriptive statistics, and they are portfolio aggregates: cohort means over the full book, with no split, no error bars, and no disclosure of how health is computed. My question: would the word-count and age gaps survive a regression that holds intent, competition, and position constant, and can the health metric be reproduced from the shipped signals? Observational cohort comparison of this shape can misorder recommendations: the older cohort may hold a different mix of intents and competition, so "older pages decline more" is measured but not necessarily causal.

**Finding #4 (CONFIRMED, fresh-page matrix / refresh result):** refreshing mature pages shows a 3.2x health step (from 10.7 to 34.5) and a 57x impression gain; at 361+ days old the ratio is 283:1 (283 growing vs 1 declining, n=284), which the paper itself calls unstable.

My methodology question: **does the validation design carry the claim?** The comparison is observational: pages were refreshed because a team decided to refresh them, and no control group is reported, so selection (the team picks pages it expects to recover) is a confound. The 3.2x and 57x numbers are two sides of the same comparison, and a 7.88:1 growth ratio plus a 283:1 row from 284 rows need a confidence statement and the denominators shown. I would track impression, click, and position change of the refreshed pages against a matched sample of untouched pages of the same age and tier, which is the same spirit as the paper's own playbook ("measure impressions, clicks, and position 30 and 60 days after the update").

One line that transfers directly to my own work: negative and mixed results are kept visible and help decision quality. If the audit finds the honest number is weaker, the weaker number stays visible.

In [1]:
import pandas as pd, numpy as np
import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

SEED = 42
print("sklearn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

df = pd.read_csv("content_refresh_anonymized.csv")
df["down"] = (df.trend_direction == "down").astype(int)   # eval only, never a feature
print("rows:", len(df), "| clients:", df.client_id.nunique())

sklearn 1.6.1 | numpy 2.0.2 | pandas 2.2.2
rows: 30000 | clients: 32


In [2]:
# Paper claims pulled from docs/flyrank-seo-research-march-2026.pdf (text extraction).
# Both lines are transcribed from the PDF. from the pdf shipped in the repo, so the notebook is reproducible.
claims = pd.DataFrame({
    "finding": ["#1 Anatomy of growing content", "#4 refresh result"],
    "paper_claim": [
        "growing pages are 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days; n=74.2K vs 45.3K)",
        "refreshing mature content shows a 3.2x health step (10.7 to 34.5) and a 57x impression gain",],
    "label_origin": [
        "direction label: rising vs falling impressions in the same portfolio window; health is a composed product score",
        "refresh = updated inside the window; outcomes = health and impression change in the same trailing window",],
    "validation_design": [
        "observational cohort means, large n, no split, no error bars, no confounder control",
        "observational; refresh is a business choice; no control group; one ratio rests on n=284 and the paper flags it",],
    "methodology_question": [
        "does the gap survive holding intent, competition, position constant; can health be reproduced from shipped signals?",
        "would the effect survive a matched control of untouched pages of the same age and tier?",],
})
print(claims[["finding", "paper_claim", "methodology_question"]].to_string(index=False))
print("\npaper's own disclosed standard: minimum bucket n = 50; negative and mixed results kept visible")

                      finding                                                                                             paper_claim                                                                                                methodology_question
#1 Anatomy of growing content growing pages are 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days; n=74.2K vs 45.3K) does the gap survive holding intent, competition, position constant; can health be reproduced from shipped signals?
            #4 refresh result             refreshing mature content shows a 3.2x health step (10.7 to 34.5) and a 57x impression gain                             would the effect survive a matched control of untouched pages of the same age and tier?

paper's own disclosed standard: minimum bucket n = 50; negative and mixed results kept visible


2. My model under an honest split (before/after)

I reran the Week-5 model using a time-aware split, training on the past window and testing on the later/current window. This is more appropriate for a trend like prediction because the model only uses information available before the period it predicts.

BEFORE — Week-5 grouped-by-client: AUC = 0.638
AFTER — time-aware split: AUC = 0.502

The model's performance dropped to approximately chance level under the time-aware split. This suggests that the Week-5 performance does not carry over when evaluated using strictly pre-label information.

In [3]:
# BEFORE: exact Week-5 reproduction
v = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
clients = v.client_id.drop_duplicates().to_numpy()
shuffled = np.random.default_rng(SEED).permutation(clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
is_test = v.client_id.isin(set(shuffled[:n_test]))
wtr, wte = v[~is_test], v[is_test]
print("grouped-by-client split (same as Week 5):", len(clients), "clients | train rows",
      len(wtr), "| test rows", len(wte), "| test base rate", round(wte.down.mean(), 3))
LOG_COLS = ["impressions_90d", "clicks_90d", "sessions_90d", "days_with_impressions",
            "word_count", "search_volume"]
RAW_COLS = ["avg_position", "ctr", "content_age_days", "days_since_last_update",
            "engagement_rate", "scroll_rate", "ai_traffic_pct"]

def make_x(d):
    x = pd.DataFrame(index=d.index)
    for c in LOG_COLS:
        x["log_" + c] = np.log1p(d[c])
    for c in RAW_COLS:
        x[c] = d[c]
    x["has_word_count"] = d.word_count.notna().astype(int)
    x["has_keyword_data"] = d.search_volume.notna().astype(int)
    for t in sorted(d.position_tier.unique()):
        x["tier_" + t] = (d.position_tier == t).astype(int)
    return x

Xtr, Xte = make_x(wtr), make_x(wte)
med = Xtr.median()
Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
ytr, yte = wtr.down.values, wte.down.values

def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return labels[order[:k]].mean()

lr_w = make_pipeline(StandardScaler(),
                     LogisticRegression(max_iter=5000, random_state=SEED)).fit(Xtr, ytr)
p_w = lr_w.predict_proba(Xte)[:, 1]

before = {"P@10": p_at_k(p_w, yte, 10), "P@20": p_at_k(p_w, yte, 20), "P@50": p_at_k(p_w, yte, 50),
          "AUC": roc_auc_score(yte, p_w), "base": float(yte.mean()), "test_n": int(len(wte))}
print("BEFORE (Week-5 grouped-by-client, features over the label window)")
print("  P@10 %.3f | P@20 %.3f | P@50 %.3f | AUC %.3f | base %.3f | test n %d"
      % (before["P@10"], before["P@20"], before["P@50"], before["AUC"],
         before["base"], before["test_n"]))

grouped-by-client split (same as Week 5): 28 clients | train rows 12939 | test rows 3787 | test base rate 0.633
BEFORE (Week-5 grouped-by-client, features over the label window)
  P@10 0.900 | P@20 0.950 | P@50 0.880 | AUC 0.638 | base 0.633 | test n 3787


In [4]:
# AFTER: time-aware split with strictly pre-label-window features.
# Recover the days 61-90 window values (impressions for the past label; clicks and sessions
# as features only): 90d total minus the two later 30-day slices. The last-30 component
# cancels exactly, so the value equals the days-61-90 data.
for m in ["impressions", "clicks", "sessions"]:
    v["early_" + m] = (v[m + "_90d"] - v[m + "_last_30d"] - v[m + "_prev_30d"]).clip(lower=0)

# Past label: the same decline rule as trend_direction, evaluated one window earlier.
v["down_past"] = ((v.impressions_prev_30d < v.early_impressions) & (v.impressions_prev_30d > 0)).astype(int)
ok_tr = (v.impressions_prev_30d > 0).values
print("past-label rows (train set):", int(ok_tr.sum()),
      "| past label rate:", round(v.down_past[ok_tr].mean(), 3))
print("agreement down vs down_past (same pages, two windows):",
      round((v.down == v.down_past)[ok_tr].mean(), 3))

STATIC = ["word_count", "search_volume", "competition", "cpc", "content_age_days"]

def make_clean(d, window):
    """window='early': features knowable before the PAST label window (days 61-90).
       window='prev':  features knowable before the CURRENT label window (days 31-60)."""
    cmap = {"early": {"clicks": "early_clicks", "sessions": "early_sessions"},
            "prev":  {"clicks": "clicks_prev_30d", "sessions": "sessions_prev_30d"}}[window]
    x = pd.DataFrame(index=d.index)
    for k, col in cmap.items():
        x["log_" + k] = np.log1p(d[col])
    for c in STATIC:
        x[c] = d[c]
    x["has_word_count"] = d.word_count.notna().astype(int)
    x["has_keyword_data"] = d.search_volume.notna().astype(int)
    for t in sorted(d.main_intent.dropna().unique()):
        x["intent_" + t] = (d.main_intent == t).astype(int)
    for t in sorted(d.competition_level.dropna().unique()):
        x["comp_" + t] = (d.competition_level == t).astype(int)
    return x

xtr2, xte2 = make_clean(v, "early"), make_clean(v, "prev")
med2 = xtr2.median()
xtr2, xte2 = xtr2.fillna(med2), xte2.fillna(med2)
y_past = v.down_past.values      # label of the PAST window
y_now = v.down.values            # label of the CURRENT window (the test target)

lr_t = make_pipeline(StandardScaler(),
                     LogisticRegression(max_iter=5000, random_state=SEED)).fit(xtr2[ok_tr], y_past[ok_tr])
p_t = lr_t.predict_proba(xte2)[:, 1]

after = {"P@10": p_at_k(p_t, y_now, 10), "P@20": p_at_k(p_t, y_now, 20), "P@50": p_at_k(p_t, y_now, 50),
         "AUC": roc_auc_score(y_now, p_t), "base": float(y_now.mean()), "test_n": int(len(y_now))}
print("\nAFTER (time-aware, strictly pre-label-window features)")
print("  P@10 %.3f | P@20 %.3f | P@50 %.3f | AUC %.3f | base %.3f | test n %d"
      % (after["P@10"], after["P@20"], after["P@50"], after["AUC"],
         after["base"], after["test_n"]))

tab = pd.DataFrame([
    {**before, "split": "grouped-by-client", "features": "over the label window"},
    {**after,  "split": "time-aware",          "features": "strictly pre-label-window"},
])
tab = tab[["split", "features", "P@10", "P@20", "P@50", "AUC"]].round(3)
print("\nBefore / after, same model class, same seed:")
print(tab.to_string(index=False))
print("base rates (random picking): BEFORE %.3f | AFTER %.3f"
      % (before["base"], after["base"]))

past-label rows (train set): 16617 | past label rate: 0.691
agreement down vs down_past (same pages, two windows): 0.583

AFTER (time-aware, strictly pre-label-window features)
  P@10 0.400 | P@20 0.450 | P@50 0.540 | AUC 0.502 | base 0.596 | test n 16726

Before / after, same model class, same seed:
            split                  features  P@10  P@20  P@50   AUC
grouped-by-client     over the label window   0.9  0.95  0.88 0.638
       time-aware strictly pre-label-window   0.4  0.45  0.54 0.502
base rates (random picking): BEFORE 0.633 | AFTER 0.596


In [ ]:
# Feature provenance: every AFTER column, its measurement window, its verdict.
prov = pd.DataFrame([
    ["log_clicks", "days 61-90 (train) / days 31-60 (test)", "legal: strictly before the label window; outside the label formula"],
    ["log_sessions", "days 61-90 (train) / days 31-60 (test)", "legal: strictly before the label window; outside the label formula"],
    ["word_count, search_volume, competition, cpc", "static snapshot metadata", "legal: not derived from the label window"],
    ["content_age_days (included)", "static snapshot metadata (creation precedes the label window)", "legal, disclosed"],
    ["has_word_count, has_keyword_data", "missingness flags, static", "legal"],
    ["intent_*, comp_*", "static category one-hots", "legal"],
    ["any impressions column", "label formula input in both windows", "EXCLUDED: label input"],
    ["*_90d, avg_position, position_tier, ctr, engagement, scroll", "window CONTAINS the last 30 days", "EXCLUDED: window overlaps label period"],
    ["*_last_30d", "the label window itself", "EXCLUDED: outcome-window inputs"],
    ["trend_direction, trend_pct", "derived from trend_pct", "EXCLUDED: label source"],
], columns=["feature family", "measurement window", "verdict"])
print(prov.to_string(index=False))

banned = ["impressions", "last_30", "trend_", "avg_position", "position_tier", "ctr",
          "engagement", "scroll_rate", "days_with", "is_declining"]
hits = [c for c in xtr2.columns if any(b in c for b in banned)]
print("\nAFTER feature columns matching a banned source:", hits)
print("every AFTER feature is strictly pre-label-window:", hits == [])

                                             feature family                                            measurement window                                                            verdict
                                                 log_clicks                        days 61-90 (train) / days 31-60 (test) legal: strictly before the label window; outside the label formula
                                               log_sessions                        days 61-90 (train) / days 31-60 (test) legal: strictly before the label window; outside the label formula
                word_count, search_volume, competition, cpc                                      static snapshot metadata                           legal: not derived from the label window
                                content_age_days (included) static snapshot metadata (creation precedes the label window)                                                   legal, disclosed
                           has_word_count, has_keyword_

Leakage audit: I checked the final feature sets for label-derived features, overlapping/future windows, and decision-derived flags. Label inputs such as trend_pct, trend_direction, and impression windows were excluded. The 90-day aggregates were also excluded from the time-aware model because they overlap the label period.

As a verification, adding the leaky impressions_last_30d feature increased Week-5 AUC from 0.638 to 0.913, and adding trend_pct increased it to 0.999. These features were therefore removed, and only the clean results are reported.

In [5]:
# The confession: add one label component, watch AUC jump, fit again without it.
def fit_auc(Xtr_f, Xte_f, ytr_f, yte_f):
    m = make_pipeline(StandardScaler(),
                     LogisticRegression(max_iter=5000, random_state=SEED)).fit(Xtr_f, ytr_f)
    return roc_auc_score(yte_f, m.predict_proba(Xte_f)[:, 1])

# Week-5 arm, same client-holdout split as before
XtrL = Xtr.copy(); XtrL["log_impressions_last_30d"] = np.log1p(wtr.impressions_last_30d)
XteL = Xte.copy(); XteL["log_impressions_last_30d"] = np.log1p(wte.impressions_last_30d)
XtrT = Xtr.copy(); XtrT["trend_pct"] = wtr.trend_pct.fillna(0)
XteT = Xte.copy(); XteT["trend_pct"] = wte.trend_pct.fillna(0)
print("Week-5 arm   clean %.3f | + impressions_last_30d %.3f | + trend_pct %.3f" % (
    roc_auc_score(yte, p_w),
    fit_auc(XtrL, XteL, ytr, yte),
    fit_auc(XtrT, XteT, ytr, yte)))

# Time arm, the same loop
xtrL2 = xtr2.copy(); xtrL2["log_impressions_prev"] = np.log1p(v.early_impressions.fillna(0))
xteL2 = xte2.copy(); xteL2["log_impressions_prev"] = np.log1p(v.impressions_prev_30d)
print("time arm     clean %.3f | + impressions_prev %.3f" % (
    after["AUC"],
    fit_auc(xtrL2[ok_tr], xteL2, y_past[ok_tr], y_now)))

print("\nafter the experiment: the clean models (0.638, 0.502) stay; the leaky variants are discarded.")

# Asserts: no label source, no product flag, no ID among either feature set.
banned = {"trend_direction", "trend_pct", "is_declining_label", "impressions_last_30d",
          "impressions_prev_30d", "health_score", "priority_score", "action_type"}
print("w05 feature set disjoint from labels, label inputs, product flags:",
      set(Xtr.columns).isdisjoint(banned))
print("time-arm feature set disjoint from the same list:",
      set(xtr2.columns).isdisjoint(banned))

Week-5 arm   clean 0.638 | + impressions_last_30d 0.913 | + trend_pct 0.999
time arm     clean 0.502 | + impressions_prev 0.583

after the experiment: the clean models (0.638, 0.502) stay; the leaky variants are discarded.
w05 feature set disjoint from labels, label inputs, product flags: True
time-arm feature set disjoint from the same list: True


On this snapshot, the Week-5 model showed higher ranking performance under the grouped-by-client split. However, when evaluated using strictly pre-label-window features in the Week-6 time-aware setup, performance was near chance (AUC 0.502 vs. 0.596 base rate). Therefore, the result should be treated as observed, directional, and decision-support only, not as evidence of future performance or causation.

In [6]:
# The rewrite's backing numbers, printed side by side.
rewrite = pd.DataFrame({
    "claim": [
        "the learned model is a better ranker at the top (P@10 0.90 vs 0.80, P@20 0.95 vs 0.85)",
        "the audit shows no measurable skill on strictly pre-label features"],
    "safe_language": [
        "observed on the client-holdout test frame (n=3,787): the logistic model ranked the top 10/20 review pages with higher precision than the rule (directional, decision-support)",
        "measured at chance with legal features (AUC 0.502 vs 0.596 base); no future-window claim is supported"],
    "supporting_number": [round(before["P@10"], 3), round(after["AUC"], 3)],
})
print(rewrite.to_string(index=False))
print("\nbase rates every claim sits next to: BEFORE %7.3f | AFTER %7.3f" % (before["base"], after["base"]))

                                                                                 claim                                                                                                                                                                 safe_language  supporting_number
the learned model is a better ranker at the top (P@10 0.90 vs 0.80, P@20 0.95 vs 0.85) observed on the client-holdout test frame (n=3,787): the logistic model ranked the top 10/20 review pages with higher precision than the rule (directional, decision-support)              0.900
                    the audit shows no measurable skill on strictly pre-label features                                                                         measured at chance with legal features (AUC 0.502 vs 0.596 base); no future-window claim is supported              0.502

base rates every claim sits next to: BEFORE   0.633 | AFTER   0.596


## Self-check

- [x] Section 1 names two paper findings and a concrete methodology question for each, constructive tone
- [x] Section 2 shows the Week-5 number and the time-aware number, same metric, base rates printed
- [x] Every AFTER feature is provably before its label window: provenance + assert in-code
- [x] Section 3 runs the whole hunt: label-derived, overlapping windows, product flags, confession experiment
- [x] Section 4 rewrites the boldest claims in careful language: observed, measured, directional, decision-support
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] The Week-5 model, features, target, and preprocessing were not edited
- [x] Committed to my repo under work/notebooks/. Done.